# Notebook 01: Khám Phá Dữ Liệu Sensor Stream (Data Exploration)

**Môn học:** Big Data  
**Đề tài:** Multiple Stream Integration with Temporal Data: Data Imputation & Prediction  

Nội dung notebook:
1. Khởi tạo & nạp dữ liệu mẫu (hoặc ICU / Airquality)
2. Phân tích độ thưa (Sparsity) và ma trận mặt nạ thiếu (Mask $M$)
3. Phân tích phân phối của từng thuộc tính (Heterogeneity)
4. Mô phỏng cơ chế chia cửa sổ thời gian (Tumbling Window) gộp thành Data Chunk

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Đưa thư mục src vào sys.path để gọi module
sys.path.append(os.path.abspath("../src"))

from data.loader import create_tumbling_windows
from visualization.plots import plot_missing_pattern

## 1. Tạo hoặc Nạp Dữ Liệu Thử Nghiệm
Mô phỏng $J=5$ sensor stream qua $N=100$ mốc thời gian với $D=6$ thuộc tính.

In [ ]:
np.random.seed(42)
J, N, D = 5, 100, 6
synthetic_data = np.random.randn(J, N, D) * 10 + 50

# Mô phỏng dữ liệu thiếu 75% (gần với ICU 80%)
mask = (np.random.rand(J, N, D) > 0.75).astype(np.float32)
data_with_missing = synthetic_data.copy()
data_with_missing[mask == 0] = np.nan

sparsity = (mask == 0).sum() / mask.size
print(f"Tensor shape: (Streams: {J}, Time: {N}, Attributes: {D})")
print(f"Tỷ lệ giá trị thiếu (Sparsity): {sparsity * 100:.2f}%")

## 2. Trực Quan Hóa Ma Trận Thiếu (Missing Pattern)

In [ ]:
# Trực quan hóa cho stream đầu tiên
plt.figure(figsize=(10, 6))
sns.heatmap(np.isnan(data_with_missing[0]), cmap="viridis", cbar=True, yticklabels=10)
plt.title("Pattern Dữ Liệu Thiếu (Màu vàng = Bị thiếu NaN)")
plt.xlabel("Thuộc tính (Attributes D)")
plt.ylabel("Thời gian (Time N)")
plt.show()

## 3. Tạo Data Chunk từ Tumbling Window
Mỗi cửa sổ độ dài $T=10$ gộp các stream thành ma trận $(J \cdot T) \times D$.

In [ ]:
T = 10
windows = create_tumbling_windows(data_with_missing, mask, window_size=T)
print(f"Số lượng tumbling window tạo được: {len(windows)}")
print(f"Kích thước data chunk đầu tiên: {windows[0][0].shape}")
print(f"Mỗi chunk có {J * T} instances và {D} attributes.")